In [1]:
!pip install -q streamlit transformers sentencepiece protobuf huggingface_hub pandas scikit-learn

import os, re, json
import numpy as np, pandas as pd
from google.colab import drive
drive.mount('/content/drive')
PROJECT_DIR = "/content/drive/MyDrive/hausa_news_project"
os.chdir(PROJECT_DIR)
os.makedirs("app", exist_ok=True)

slug = lambda s: re.sub(r"\W+", "_", s.strip())
print("best_transformer files:", os.listdir("best_transformer") if os.path.exists("best_transformer") else "MISSING, re-run notebook 04")
meta = json.load(open("best_transformer/meta.json"))
print(json.dumps(meta, indent=2))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 67.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 86.0 MB/s eta 0:00:00
Mounted at /content/drive
best_transformer files: ['tokenizer.json', 'config.json', 'meta.json', 'tokenizer_config.json', 'model.safetensors']
{
  "val_f1": 0.916803401307814,
  "run": "AfroXLMR-base | head256 | lr1e-05",
  "model_id": "Davlan/afro-xlmr-base",
  "max_len": 256,
  "strategy": "head",
  "lr": 1e-05,
  "epochs_run": 6,
  "batch_size": 16,
  "grad_accum": 1,
  "seed": 42
}


In [2]:
%%writefile app/requirements.txt
--extra-index-url https://download.pytorch.org/whl/cpu
torch
transformers
sentencepiece
protobuf
huggingface_hub
streamlit
pandas
numpy

Writing app/requirements.txt


In [3]:
%%writefile app/classifier.py
import json, os, re
import numpy as np
import torch
from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer, AutoModelForSequenceClassification

TAIL_FRAC = 0.25


def clean_text(t):
    """Identical to the cleaning used when the dataset was prepared."""
    t = re.sub(r"http\S+|www\.\S+", " ", t)
    return re.sub(r"\s+", " ", t).strip()


class HausaNewsClassifier:
    """Loads the fine-tuned model (Hub id or local folder) and classifies Hausa news text."""

    def __init__(self, model_path):
        if os.path.isdir(model_path):
            meta_file = os.path.join(model_path, "meta.json")
        else:
            meta_file = hf_hub_download(model_path, "meta.json")
        self.meta = json.load(open(meta_file))
        self.max_len = int(self.meta["max_len"])
        self.strategy = self.meta["strategy"]          # "head" or "head+tail", must match training
        self.tok = AutoTokenizer.from_pretrained(model_path)
        self.model = AutoModelForSequenceClassification.from_pretrained(
            model_path, low_cpu_mem_usage=True).eval()
        self.labels = [self.model.config.id2label[i] for i in range(self.model.config.num_labels)]
        t = self.tok
        self.start = t.cls_token_id if t.cls_token_id is not None else t.bos_token_id
        self.end = t.sep_token_id if t.sep_token_id is not None else t.eos_token_id

    def encode(self, text):
        ids = self.tok(clean_text(text), add_special_tokens=False, truncation=False)["input_ids"]
        n_total, budget = len(ids), self.max_len - 2
        if n_total > budget:
            if self.strategy == "head":
                ids = ids[:budget]
            else:
                tail = int(budget * TAIL_FRAC)
                ids = ids[:budget - tail] + ids[-tail:]
        return [self.start] + ids + [self.end], n_total, budget

    @torch.inference_mode()
    def predict_probs(self, text):
        ids, n_total, budget = self.encode(text)
        out = self.model(input_ids=torch.tensor([ids]),
                         attention_mask=torch.ones(1, len(ids), dtype=torch.long))
        return torch.softmax(out.logits[0].float(), -1).numpy(), n_total, budget

Writing app/classifier.py


In [4]:
%%writefile app/streamlit_app.py
import numpy as np
import pandas as pd
import streamlit as st
from classifier import HausaNewsClassifier

MODEL_ID = "Batonicarla1/hausa-news-classifier"

st.set_page_config(page_title="Hausa News Topic Classifier", page_icon="📰")


@st.cache_resource(show_spinner="Loading the model (the first start takes a minute)...")
def get_classifier():
    return HausaNewsClassifier(MODEL_ID)


clf = get_classifier()

EXAMPLES = {
    "Health": "Lafiya Zinariya: Halin da ma'aikatan lafiya ke ciki a annobar cutar korona. Ku latsa lasifikar d ake sama don ssauraron cikakken shirin wanda Habiba Adamu ta gabatar. Wani bincike da aka wallafa a mujallar kiwon lafiya ta The Lancet ya gano cewa, ma'aikatan kiwon lafiya na fuskantar karin barazanar kamuwa da cutar Covid-19.",
    "Politics": "Kwankwaso ba shi da alkibla – Ganduje. Gwamnan jihar Kano da ke Najeriya, Abdullahi Umar Ganduje, ya bayyana tsohon gwamnan jihar Sanata Rabi'u Musa Kwankwaso a matsayin dan siyasa 'mara alkibla' wanda ba shi da 'tsari'.",
    "Technology": "Amurka na bincike kan wasu abubuwa da ke yawo a sararin samaniyarta. Wani rahoton gwamnatin Amurka kan ganin wasu abubuwa masu tashi sama da ba a san ko meye ba (UFOs), ya gano cewa babu wata shaidar cewa akwai wasu halittu na 'alien' amma ba su cire tsammanin akwai ba.",
    "Failure case (business, predicted religion)": "Asarar da Saudiyya ke yi saboda rashin mahajjata daga kasashen duniya. Saudiyya na daga cikin manyan kasashen duniya da annobar cutar korona ta haifar da koma baya gahanyoyin samun kudaden shigarta. Saudiyya na samum ɗumbin kudaden shiga a lokutan ayyukan Hajji da Umara da miliyoyin Musulmai kan halarta a duk shekara daga kasashen duniya.",
}

st.title("Hausa News Topic Classifier")
st.write("Classifies Hausa news text into 7 topics: business, entertainment, health, politics, "
         "religion, sports, technology. Fine-tuned on MasakhaNEWS (BBC Hausa). It only knows these "
         "7 topics and was trained on news articles, not social media.")

if "text" not in st.session_state:
    st.session_state["text"] = ""
choice = st.selectbox("Try an example", ["(none)"] + list(EXAMPLES))
if st.button("Load example", key="load") and choice != "(none)":
    st.session_state["text"] = EXAMPLES[choice]

text = st.text_area("Hausa news text (headline and/or article)", key="text", height=220,
                    placeholder="Paste a Hausa news headline or article here...")

if st.button("Classify", type="primary", key="classify"):
    if not text or len(text.split()) < 3:
        st.warning("Please enter a Hausa news headline or article (at least a few words).")
    else:
        probs, n_total, budget = clf.predict_probs(text)
        top = int(np.argmax(probs))
        st.success(f"Predicted topic: {clf.labels[top]} (confidence {probs[top]:.0%})")
        if probs[top] < 0.5:
            st.info("Low confidence: the text may mix topics or fall outside these categories.")
        if len(text.split()) < 10:
            st.info("Very short input: the model was trained on full news articles, so short text is less reliable.")
        if n_total > budget:
            st.info(f"Long text: only {budget} of {n_total} tokens were used ({clf.strategy} truncation).")
        st.bar_chart(pd.DataFrame({"probability": probs}, index=clf.labels))

with st.expander("About this model"):
    st.write("Fine-tuned from Davlan/afro-xlmr-base on the Hausa part of MasakhaNEWS. "
             "Model page: https://huggingface.co/Batonicarla1/hausa-news-classifier")

Writing app/streamlit_app.py


In [5]:
import sys, importlib
sys.path.insert(0, "app")
import classifier
importlib.reload(classifier)
clf = classifier.HausaNewsClassifier("best_transformer")

test = pd.read_csv("test.csv")
saved_logits = np.load("preds/" + slug(meta["run"]) + ".npz")["test_logits"]
saved = saved_logits.argmax(1)

probs, n_total, budget = clf.predict_probs(test["text_clean"][0])
print("Example 0:", {clf.labels[i]: round(float(probs[i]), 3) for i in np.argsort(-probs)[:3]})

N = 150
live = np.array([int(np.argmax(clf.predict_probs(t)[0])) for t in test["text_clean"][:N]])
print(f"Agreement with saved predictions: {(live == saved[:N]).mean():.1%} on {N} articles")
print("Live accuracy on these:", round((live == test['label_id'][:N].values).mean(), 3))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1040 > 512). Running this sequence through the model will result in indexing errors


Example 0: {'business': 0.982, 'politics': 0.004, 'health': 0.004}
Agreement with saved predictions: 100.0% on 150 articles
Live accuracy on these: 0.82


In [6]:
y = test["label_id"].values
pr = np.exp(saved_logits - saved_logits.max(1, keepdims=True)); pr /= pr.sum(1, keepdims=True)
pred, conf = pr.argmax(1), pr.max(1)
correct, wrong_ = np.where(pred == y)[0], np.where(pred != y)[0]

show = []
for lab in ["sports", "health", "politics", "technology"]:
    c = clf.labels.index(lab)
    show += [i for i in correct[np.argsort(-conf[correct])] if y[i] == c][:1]
show += list(wrong_[np.argsort(-conf[wrong_])][:2])
for i in show:
    print(f"[{i}] true={clf.labels[y[i]]} pred={clf.labels[pred[i]]} p={conf[i]:.2f}\n{test['text_clean'][i][:350]}\n")

[517] true=sports pred=sports p=0.99
H﻿ernandez na Faransa ya gama buga Gasar Kofin Duniya a Qatar. M﻿ai tsaron bayan Faransa, Lukas Hernandez ya gama buga Gasar Cin Kofin Duniya a Qatar, bayan da ya ji rauni a wasa da Australia ranar Talata. M﻿ai shekara 26, mai taka leda a Bayern Munich ya ji rauni a lokacin da ya fadi da gwiwar kafa daf da Australia za ta ci kwallo a wasan da aka d

[273] true=health pred=health p=0.99
Lafiya Zinariya: Halin da ma'aikatan lafiya ke ciki a annobar cutar korona. Ku latsa lasifikar d ake sama don ssauraron cikakken shirin wanda Habiba Adamu ta gabatar. Wani bincike da aka wallafa a mujallar kiwon lafiya ta The Lancet ya gano cewa, ma'aikatan kiwon lafiya na fuskantar karin barazanar kamuwa da cutar Covid-19. Sai dai hukumomi a fadin

[309] true=politics pred=politics p=0.99
Kwankwaso ba shi da alkibla – Ganduje. Gwamnan jihar Kano da ke Najeriya, Abdullahi Umar Ganduje, ya bayyana tsohon gwamnan jihar Sanata Rabi'u Musa Kwankwaso a matsayin dan siyasa 

In [7]:
from huggingface_hub import notebook_login
notebook_login()     # paste your WRITE token in the box

In [8]:
from huggingface_hub import HfApi
from sklearn.metrics import accuracy_score, f1_score

HF_USER = "Batonicarla1"
MODEL_REPO = f"{HF_USER}/hausa-news-classifier"

d = np.load("preds/" + slug(meta["run"]) + ".npz")
tp, yt = d["test_logits"].argmax(1), pd.read_csv("test.csv")["label_id"].values
acc = round(accuracy_score(yt, tp), 4)
f1m = round(f1_score(yt, tp, average="macro"), 4)
print("Test accuracy:", acc, "| Test macro-F1:", f1m)

card = f"""---
language: ha
tags: [text-classification, hausa, masakhanews, african-languages]
datasets: [masakhane/masakhanews]
base_model: {meta['model_id']}
metrics: [accuracy, f1]
---

# Hausa news topic classifier

Fine-tuned from `{meta['model_id']}` on the Hausa portion of MasakhaNEWS (Adelani et al., 2023) to
predict one of 7 topics: business, entertainment, health, politics, religion, sports, technology.

## Results (official MasakhaNEWS Hausa test split, 637 articles)
- Accuracy: {acc}
- Macro-F1: {f1m}

## Training
- Run: {meta['run']}; max length {meta['max_len']} tokens; truncation: {meta['strategy']}
- Learning rate {meta['lr']}, batch size {meta['batch_size']} x {meta['grad_accum']}; model selected on validation macro-F1.

## Limitations
- Trained on BBC Hausa news only; expect lower accuracy on social media, other news sites, or other dialects/spelling conventions.
- Knows only the 7 topics above and has no "none of these" class.
- Small training set (about 2.2k articles); the test set has 637 articles, so small differences between models are within noise.

## Intended use
Educational / research demonstration. Check the base model's card and the dataset card for license terms before any other use.
"""
api = HfApi()
api.create_repo(MODEL_REPO, repo_type="model", exist_ok=True)
api.upload_folder(folder_path="best_transformer", repo_id=MODEL_REPO, repo_type="model")
api.upload_file(path_or_fileobj=card.encode(), path_in_repo="README.md",
                repo_id=MODEL_REPO, repo_type="model")
print("Model:", f"https://huggingface.co/{MODEL_REPO}")

Test accuracy: 0.9058 | Test macro-F1: 0.8981


No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


Model: https://huggingface.co/Batonicarla1/hausa-news-classifier


In [9]:
from huggingface_hub import hf_hub_download
hub_meta = json.load(open(hf_hub_download(MODEL_REPO, "meta.json", force_download=True)))
print("Hub model run:", hub_meta["run"], "| local best:", meta["run"])
assert hub_meta["run"] == meta["run"], "Hub model differs from your best model, re-run Cell 8"

meta.json:   0%|          | 0.00/223 [00:00<?, ?B/s]

Hub model run: AfroXLMR-base | head256 | lr1e-05 | local best: AfroXLMR-base | head256 | lr1e-05


In [11]:
import os
from streamlit.testing.v1 import AppTest
sys.path.insert(0, "app")

# Use absolute path to resolve the file correctly from the Colab notebook environment
app_path = os.path.abspath("app/streamlit_app.py")
at = AppTest.from_file(app_path, default_timeout=900).run()
assert not at.exception, at.exception

for name, expected in [("Health", "health"), ("Politics", "politics"), ("Technology", "technology"),
                       ("Failure case (business, predicted religion)", "religion")]:
    at.selectbox[0].select(name).run()
    at.button(key="load").click().run()
    at.button(key="classify").click().run()
    print(f"{name:45s} -> {at.success[0].value}")
    assert expected in at.success[0].value.lower()
print("App test passed")

2026-10-13 09:10:46.342 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


config.json:   0%|          | 0.00/1.09k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/343 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Health                                        -> Predicted topic: health (confidence 99%)
Politics                                      -> Predicted topic: politics (confidence 98%)
Technology                                    -> Predicted topic: technology (confidence 96%)
Failure case (business, predicted religion)   -> Predicted topic: religion (confidence 99%)
App test passed
